In [6]:
%pip install pandas rapidfuzz

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:

import pandas as pd

df = pd.read_csv("student_admission_prediction.csv")

print("Dataset loaded successfully!")
print(df.head())


Dataset loaded successfully!
   Entrance_Exam_Score  Undergraduate_GPA Work_Experience  Admission_Status
0                   78               7.56             Yes                 0
1                   91               7.51              No                 0
2                   68               8.99             Yes                 0
3                   54               8.25             Yes                 0
4                   82               8.51             Yes                 0


In [8]:

print("Dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nDataset information:")
df.info()

print("\nMissing values:")
print(df.isnull().sum())

print("\nExact duplicate rows:")
print(df.duplicated().sum())


Dataset shape: (500, 4)

Column names:
['Entrance_Exam_Score', 'Undergraduate_GPA', 'Work_Experience', 'Admission_Status']

Dataset information:
<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Entrance_Exam_Score  500 non-null    int64  
 1   Undergraduate_GPA    500 non-null    float64
 2   Work_Experience      500 non-null    str    
 3   Admission_Status     500 non-null    int64  
dtypes: float64(1), int64(2), str(1)
memory usage: 17.0 KB

Missing values:
Entrance_Exam_Score    0
Undergraduate_GPA      0
Work_Experience        0
Admission_Status       0
dtype: int64

Exact duplicate rows:
1


In [ ]:


clean_df = df.copy()


clean_df["Work_Experience"] = (
    clean_df["Work_Experience"]
    .astype("string")
    .str.strip()
    .str.title()
)


before = len(clean_df)


clean_df = clean_df.drop_duplicates().copy()

after = len(clean_df)

print("Records before cleaning:", before)
print("Records after cleaning:", after)
print("Duplicates removed:", before - after)


Records before cleaning: 500
Records after cleaning: 499
Duplicates removed: 1


In [10]:

duplicate_rows = df[df.duplicated(keep=False)]

print("Duplicate records:")
display(duplicate_rows)


Duplicate records:


,Entrance_Exam_Score,Undergraduate_GPA,Work_Experience,Admission_Status
115,99,9.03,Yes,0
344,99,9.03,Yes,0


In [ ]:



candidates = []

for i in range(len(clean_df)):
    for j in range(i + 1, len(clean_df)):

        row1 = clean_df.iloc[i]
        row2 = clean_df.iloc[j]

        score_diff = abs(
            row1["Entrance_Exam_Score"]
            - row2["Entrance_Exam_Score"]
        )

        gpa_diff = abs(
            row1["Undergraduate_GPA"]
            - row2["Undergraduate_GPA"]
        )

        same_experience = (
            row1["Work_Experience"] == row2["Work_Experience"]
        )

        same_status = (
            row1["Admission_Status"] == row2["Admission_Status"]
        )

        if (
            score_diff <= 2
            and gpa_diff <= 0.15
            and same_experience
            and same_status
        ):
            candidates.append({
                "Row_1": i,
                "Row_2": j,
                "Score_Difference": score_diff,
                "GPA_Difference": round(gpa_diff, 3),
                "Work_Experience": row1["Work_Experience"],
                "Admission_Status": row1["Admission_Status"]
            })

candidate_df = pd.DataFrame(candidates)

print("Possible similar record pairs:", len(candidate_df))
display(candidate_df.head(20))


Possible similar record pairs: 326


,Row_1,Row_2,Score_Difference,GPA_Difference,Work_Experience,Admission_Status
0,0,482,1,0.11,Yes,0
1,2,119,0,0.11,Yes,0
2,4,107,2,0.03,Yes,0
3,4,335,1,0.06,Yes,0
4,4,355,2,0.06,Yes,0
5,4,444,2,0.14,Yes,0
6,5,147,0,0.03,No,0
7,5,233,2,0.13,No,0
8,5,337,1,0.02,No,0
9,5,484,1,0.04,No,0


In [12]:

clean_df.to_csv("cleaned_admissions.csv", index=False)

print("Cleaned dataset saved successfully!")
print("Total clean records:", len(clean_df))


Cleaned dataset saved successfully!
Total clean records: 499


In [13]:

print("===== UNIVERSITY ADMISSIONS REPORT =====")

print("Original records:", len(df))
print("Clean records:", len(clean_df))
print("Exact duplicates removed:", len(df) - len(clean_df))
print("Missing values in original data:", int(df.isnull().sum().sum()))
print("Possible similar record pairs:", len(candidate_df))

print("\nFinal columns:")
print(clean_df.columns.tolist())

print("\nCleaned data preview:")
display(clean_df.head())

print("\nReport completed!")


===== UNIVERSITY ADMISSIONS REPORT =====
Original records: 500
Clean records: 499
Exact duplicates removed: 1
Missing values in original data: 0
Possible similar record pairs: 326

Final columns:
['Entrance_Exam_Score', 'Undergraduate_GPA', 'Work_Experience', 'Admission_Status']

Cleaned data preview:


,Entrance_Exam_Score,Undergraduate_GPA,Work_Experience,Admission_Status
0,78,7.56,Yes,0
1,91,7.51,No,0
2,68,8.99,Yes,0
3,54,8.25,Yes,0
4,82,8.51,Yes,0



Report completed!
